In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install torch torchvision transformers pandas tqdm pillow accelerate openpyxl

In [ ]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from PIL import Image

from transformers import Blip2Processor, Blip2ForConditionalGeneration

**Load BLIP Model**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/10.0G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Blip2ForConditionalGeneration(
  (vision_model): Blip2VisionModel(
    (embeddings): Blip2VisionEmbeddings(
      (patch_embedding): Conv2d(3, 1408, kernel_size=(14, 14), stride=(14, 14))
    )
    (encoder): Blip2Encoder(
      (layers): ModuleList(
        (0-38): 39 x Blip2EncoderLayer(
          (self_attn): Blip2Attention(
            (qkv): Linear(in_features=1408, out_features=4224, bias=True)
            (projection): Linear(in_features=1408, out_features=1408, bias=True)
          )
          (layer_norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
          (mlp): Blip2MLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=1408, out_features=6144, bias=True)
            (fc2): Linear(in_features=6144, out_features=1408, bias=True)
          )
          (layer_norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
  )
  (qf

**Dataset Configuration**

In [ ]:
DATASET_ROOT = "/content/drive/MyDrive/Navarasa_Dataset"
OUTPUT_EXCEL = "navarasa_image_text_correct.xlsx"

NAVARASA_LIST = [
    "Adbhuta",
    "Bhayanaka",
    "Bibhatsa",
    "Hasya",
    "Karuna",
    "Raudra",
    "Shantha",
    "Shringara",
    "Veera"
]

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp")

**IMAGE → RAW CAPTION**

In [ ]:
def generate_raw_caption(image_path):
    image = Image.open(image_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False
        )

    caption = processor.decode(output[0], skip_special_tokens=True)
    return caption

**Inject Navarasa**

In [ ]:
def navarasa_description(caption, navarasa):
    return (
        f"The subject exhibiting {navarasa}, "
        f"with {caption}, "
        f"conveying the emotional essence of {navarasa}."
    )

**Intensity**

In [ ]:
def infer_intensity(caption):
    c = caption.lower()

    if any(w in c for w in ["wide", "tense", "rigid", "open mouth", "staring"]):
        return "high"
    elif any(w in c for w in ["soft", "slight", "gentle", "relaxed"]):
        return "low"
    else:
        return "medium"

**Folders -> Excel Rows**

In [ ]:
rows = []

for navarasa in tqdm(NAVARASA_LIST, desc="Processing Navarasas"):
    folder_path = os.path.join(DATASET_ROOT, navarasa)

    if not os.path.isdir(folder_path):
        continue

    for img_name in os.listdir(folder_path):
        if not img_name.lower().endswith(VALID_EXTENSIONS):
            continue

        image_path = os.path.join(folder_path, img_name)
        relative_path = f"{navarasa}/{img_name}"

        try:
            raw_caption = generate_raw_caption(image_path)
            description = navarasa_description(raw_caption, navarasa)
            intensity = infer_intensity(raw_caption)
        except Exception as e:
            print(f"❌ Error processing {relative_path}: {e}")
            continue

        rows.append({
            "image_path": relative_path,
            "navarasa": navarasa,
            "intensity": intensity,
            "description": description
        })

Processing Navarasas: 100%|██████████| 9/9 [1:31:38<00:00, 610.96s/it]


**Save to Excel**

In [ ]:
df = pd.DataFrame(rows)
df.to_excel(OUTPUT_EXCEL, index=False)

print("✅ Dataset created:", OUTPUT_EXCEL)
print("Total samples:", len(df))

✅ Dataset created: navarasa_image_text_correct.xlsx
Total samples: 14003
